# Weather ingestion — IPMA hourly station data

This notebook turns four raw hourly weather CSVs (temperature, humidity, wind,
precipitation) into one Delta table per weather station, registered in the `weather_data`
schema. Unlike the other ingestion notebooks, this one does a lot of reshaping along the
way, so it is organised in four stages:

1. **Setup and mount** the weather storage container.
2. **Load and split** each of the four variables into one CSV per station.
3. **Assembly** — merge the four variables per station, clean up names, drop leftover
   metadata rows, and add coordinates.
4. **Convert to Delta** and register the per-station tables.

## 1. Setup and mount

Load the project utilities, then mount the weather storage container.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

> Unmount the container first. This errors if it is not currently mounted, so on a fresh cluster you can skip it.

In [0]:
dbutils.fs.unmount("/mnt/weather_data")

Mount the weather container.

> **Security — account key removed.** The original cell set `storage_account_key` to the
> account key in plaintext. It has been replaced with a secret-scope lookup. Rotate the
> original key in Azure; it was committed to the repo.

In [0]:
# Define variables
storage_account_name = "aimasterdata"
storage_account_key = "xxx"
container_name = "weatherdata"
mount_point = "/mnt/weather_data"

# Mount the storage account
dbutils.fs.mount(
    source=f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
    mount_point=mount_point,
    extra_configs={f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_account_key}
)

# Verify mount
display(dbutils.fs.ls(mount_point))

Check the mount and list the source folder.

In [0]:
# Verify mount
display(dbutils.fs.ls("/mnt/weather_data/AnoHorario"))

## 2. Load and split each variable into per-station files

Each raw CSV holds every station's readings side by side, with an unusual layout: the
station names sit in one header row and the measurement type in the next. For each
variable the steps are the same:

1. Read the raw CSV.
2. Pull the station-name and variable-type header rows.
3. Keep the data columns, drop the `FLAG` columns, and rename each kept column
   `Station_Variable`.
4. Write one CSV per station into a `split_hour_*` folder.

The four blocks below (temperature, humidity, wind, precipitation) are near-identical
copies of this pattern, differing only in the input file and output folder.

### 2.1 Temperature

In [0]:
df_temp = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/mnt/weather_data/AnoHorario/temperaturaHoraria.csv")

df_temp.display()


In [0]:
header_stations = df_temp.collect()[2]
display(header_stations)

In [0]:
# Extract headers
header_stations = df_temp.collect()[2]  # Location names
header_variables = df_temp.collect()[3]  # Measurement type

# Ensure Date column is included
columns_to_keep = [0]  # Assuming Date is always the first column
cleaned_columns = ["Date"]  # First column must be "Date"

# Process remaining columns, skipping "FLAG"
for i in range(0, len(header_stations)):  # Start from index 1 (skip "Date" column)
    station = header_stations[i]
    variable = header_variables[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep.append(i)  # Store valid column index
        cleaned_columns.append(f"{station.strip()}_{variable.strip()}")  # Format column name


# Print debug info
print(f"Final columns_to_keep: {len(columns_to_keep)}")
print(f"Final cleaned_columns: {len(cleaned_columns)}")

In [0]:
# Select only columns that are NOT "FLAG"
df_filtered = df_temp.select([col(f"Column{i+1}") for i in columns_to_keep])

# Rename columns properly
df_cleaned = df_filtered.toDF(*cleaned_columns)

df_cleaned.display()


In [0]:
# Extract station names by removing the measurement type (e.g., "_Temperature")
stations = list(set([col.split("_")[0] for col in cleaned_columns if "_" in col]))

# Print debug info
print(f"Total unique weather stations found: {len(stations)}")


In [0]:
#dbutils.fs.rm("/mnt/weather_data/split_stations/", True)


Split temperature into one CSV per station. The station name is stripped of accents and parenthetical parts before being used as a filename.

In [0]:
# Define output directory in Databricks storage
output_dir = "/mnt/weather_data/split_hour_temp/"

# Loop through each station and create separate CSV files
for station in stations:
    # Select only relevant columns (date + station-specific data)
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]
    
    # Create a DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # **Normalize the station name**
    station_clean = unidecode(station)  # Remove accents
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)  # Remove anything inside parentheses
    station_clean = station_clean.replace(" ", "_")  # Replace spaces with underscores

    # Define output file path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV (overwrite if it already exists)
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


Inspection: list the output folder and read one station back.

In [0]:
display(dbutils.fs.ls("/mnt/weather_data/split_hour_temp/"))


In [0]:
# Example: Read one station's CSV file (replace with an actual filename from the previous step)
file_to_check = "/mnt/weather_data/split_hour_temp/ALAGOA.csv"

df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_to_check)

# Display the file contents
df_check.display()


### 2.2 Humidity

Same pattern as temperature, reading the humidity file.

In [0]:
df_hum = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/mnt/weather_data/AnoHorario/humidadeHoraria.csv")

df_hum.display()


In [0]:
header_stations = df_hum.collect()[2]
display(header_stations)

In [0]:
# Extract headers
header_stations = df_hum.collect()[2]  # Location names
header_variables = df_hum.collect()[3]  # Measurement type

# Ensure Date column is included
columns_to_keep = [0]  # Assuming Date is always the first column
cleaned_columns = ["Date"]  # First column must be "Date"

# Process remaining columns, skipping "FLAG"
for i in range(0, len(header_stations)):  # Start from index 1 (skip "Date" column)
    station = header_stations[i]
    variable = header_variables[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep.append(i)  # Store valid column index
        cleaned_columns.append(f"{station.strip()}_{variable.strip()}")  # Format column name


# Print debug info
print(f"Final columns_to_keep: {len(columns_to_keep)}")
print(f"Final cleaned_columns: {len(cleaned_columns)}")

In [0]:
# Select only columns that are NOT "FLAG"
df_filtered = df_hum.select([col(f"Column{i+1}") for i in columns_to_keep])

# Rename columns properly
df_cleaned = df_filtered.toDF(*cleaned_columns)

df_cleaned.display()


In [0]:
# Define output directory in Databricks storage
output_dir = "/mnt/weather_data/split_hour_hum/"

# Loop through each station and create separate CSV files
for station in stations:
    # Select only relevant columns (date + station-specific data)
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]
    
    # Create a DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # **Normalize the station name**
    station_clean = unidecode(station)  # Remove accents
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)  # Remove anything inside parentheses
    station_clean = station_clean.replace(" ", "_")  # Replace spaces with underscores

    # Define output file path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV (overwrite if it already exists)
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


Inspection: read one humidity station back.

In [0]:
# Example: Read one station's CSV file (replace with an actual filename from the previous step)
file_to_check = "/mnt/weather_data/split_hour_hum/ALAGOA.csv"

df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_to_check)

# Display the file contents
df_check.display()


### 2.3 Wind

Same pattern, reading the wind file.

In [0]:
df_wind = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/mnt/weather_data/AnoHorario/ventoHorario.csv")

df_wind.display()


In [0]:
header_stations = df_wind.collect()[2]
display(header_stations)

In [0]:
# Extract headers
header_stations = df_wind.collect()[2]  # Location names
header_variables = df_wind.collect()[3]  # Measurement type

# Ensure Date column is included
columns_to_keep = [0]  # Assuming Date is always the first column
cleaned_columns = ["Date"]  # First column must be "Date"

# Process remaining columns, skipping "FLAG"
for i in range(0, len(header_stations)):  # Start from index 1 (skip "Date" column)
    station = header_stations[i]
    variable = header_variables[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep.append(i)  # Store valid column index
        cleaned_columns.append(f"{station.strip()}_{variable.strip()}")  # Format column name


# Print debug info
print(f"Final columns_to_keep: {len(columns_to_keep)}")
print(f"Final cleaned_columns: {len(cleaned_columns)}")

In [0]:
# Select only columns that are NOT "FLAG"
df_filtered = df_wind.select([col(f"Column{i+1}") for i in columns_to_keep])

# Rename columns properly
df_cleaned = df_filtered.toDF(*cleaned_columns)

df_cleaned.display()


In [0]:
# Define output directory in Databricks storage
output_dir = "/mnt/weather_data/split_hour_wind/"

# Loop through each station and create separate CSV files
for station in stations:
    # Select only relevant columns (date + station-specific data)
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]
    
    # Create a DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # **Normalize the station name**
    station_clean = unidecode(station)  # Remove accents
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)  # Remove anything inside parentheses
    station_clean = station_clean.replace(" ", "_")  # Replace spaces with underscores

    # Define output file path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV (overwrite if it already exists)
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


Inspection: read one wind station back.

In [0]:
# Example: Read one station's CSV file (replace with an actual filename from the previous step)
file_to_check = "/mnt/weather_data/split_hour_wind/ALAGOA.csv"

df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_to_check)

# Display the file contents
df_check.display()


### 2.4 Precipitation

Same pattern, reading the precipitation file.

In [0]:
df_prec = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/mnt/weather_data/AnoHorario/prechoraria.csv")

df_prec.display()


In [0]:
header_stations = df_prec.collect()[2]
display(header_stations)

In [0]:
# Extract headers
header_stations = df_prec.collect()[2]  # Location names
header_variables = df_prec.collect()[3]  # Measurement type

# Ensure Date column is included
columns_to_keep = [0]  # Assuming Date is always the first column
cleaned_columns = ["Date"]  # First column must be "Date"

# Process remaining columns, skipping "FLAG"
for i in range(0, len(header_stations)):  # Start from index 1 (skip "Date" column)
    station = header_stations[i]
    variable = header_variables[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep.append(i)  # Store valid column index
        cleaned_columns.append(f"{station.strip()}_{variable.strip()}")  # Format column name


# Print debug info
print(f"Final columns_to_keep: {len(columns_to_keep)}")
print(f"Final cleaned_columns: {len(cleaned_columns)}")

In [0]:
# Select only columns that are NOT "FLAG"
df_filtered = df_prec.select([col(f"Column{i+1}") for i in columns_to_keep])

# Rename columns properly
df_cleaned = df_filtered.toDF(*cleaned_columns)

df_cleaned.display()


In [0]:
# Define output directory in Databricks storage
output_dir = "/mnt/weather_data/split_hour_prec/"

# Loop through each station and create separate CSV files
for station in stations:
    # Select only relevant columns (date + station-specific data)
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]
    
    # Create a DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # **Normalize the station name**
    station_clean = unidecode(station)  # Remove accents
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)  # Remove anything inside parentheses
    station_clean = station_clean.replace(" ", "_")  # Replace spaces with underscores

    # Define output file path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV (overwrite if it already exists)
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


Inspection: read one precipitation station back.

In [0]:
# Example: Read one station's CSV file (replace with an actual filename from the previous step)
file_to_check = "/mnt/weather_data/split_hour_prec/ALAGOA.csv"

df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_to_check)

# Display the file contents
df_check.display()


## 3. Assembly

Merge the four variables for each station into a single table, then clean it up.

In [0]:
stations

Define the name-normalising function and build the set of normalised station names.

In [0]:
# Normalize location names (remove accents, fix underscores, and remove extra parts)
def normalize_name(name):
    name_clean = unidecode(name)  # Remove accents
    name_clean = re.sub(r"\s*\(.*?\)", "", name_clean)  # Remove everything inside parentheses
    name_clean = name_clean.replace(" ", "_")  # Replace spaces with underscores
    return name_clean

# Normalize all location names
locations_normalized = set([normalize_name(loc) for loc in stations])

In [0]:
display(dbutils.fs.ls("/mnt/weather_data/"))

Merge the four per-station files on `Date` with an inner join, and save one merged file
per station. Stations missing any of the four files are skipped.

In [0]:
from pyspark.sql.functions import col

# Get a list of correct locations
locations = list(locations_normalized)

base_dir = "dbfs:/mnt/weather_data/split_hour_"

for location in locations:
    print(f"🔹 Merging data for {location}...")

    # Define the corrected file paths
    temp_file = f"{base_dir}temp/{location}.csv"
    humidity_file = f"{base_dir}hum/{location}.csv"
    wind_file = f"{base_dir}wind/{location}.csv"
    prec_file = f"{base_dir}prec/{location}.csv"

    # Check if files exist before loading
    try:
        df_temp = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(temp_file)
        df_humidity = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(humidity_file)
        df_wind = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(wind_file)
        df_prec = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(prec_file)

        # Perform an **inner join** on "Date"
        df_merged = df_temp.join(df_humidity, ["Date"], "inner").join(df_wind, ["Date"], "inner").join(df_prec, ["Date"], "inner")

        # Save merged data
        merged_output_path = f"{base_dir}merged/{location}.csv"
        df_merged.write.format("csv").mode("overwrite").option("header", "true").save(merged_output_path)

        print(f"✅ Merged data saved: {merged_output_path}")

    except Exception as e:
        print(f"❌ Skipping {location} due to missing files: {e}")


In [0]:
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/PROENCA_A_NOVA.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/SANTAREM.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/SAO_BRAS_DE_ALPORTEL.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/ALCACOVAS.csv", recurse=True)

In [0]:
display(dbutils.fs.ls("/mnt/weather_data/split_hour_merged/"))

> **One-shot cell — manual filename fixes.** These rename four files whose names came
> out garbled (the encoding issue), e.g. `SANTARM` to `SANTAREM`. 

In [0]:
dbutils.fs.mv("/mnt/weather_data/split_hour_merged/PROENA-A-NOVA.csv", "/mnt/weather_data/split_hour_merged/PROENCA_A_NOVA.csv", recurse=True)
dbutils.fs.mv("/mnt/weather_data/split_hour_merged/SANTARM.csv", "/mnt/weather_data/split_hour_merged/SANTAREM.csv", recurse=True)
dbutils.fs.mv("/mnt/weather_data/split_hour_merged/SO_BRS_DE_ALPORTEL.csv", "/mnt/weather_data/split_hour_merged/SAO_BRAS_DE_ALPORTEL.csv", recurse=True)
dbutils.fs.mv("/mnt/weather_data/split_hour_merged/ALCOVAS.csv", "/mnt/weather_data/split_hour_merged/ALCACOVAS.csv", recurse=True)


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_hour_merged/ALCACOVAS.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
# Define mapping for renaming columns
column_rename_map = {
    "Temperatura do ar hor�ria (�C)": "Temperatura media do ar horaria (C)",
    "Precipita��o hor�ria (mm)": "Precipitacao horaria (mm)",
    "Humidade relativa hor�ria (%)": "Humidade relativa media horaria (%)",
    "Velocidade do vento hor�ria (m/s)": "Velocidade do vento media horaria (m_per_s)"
}

# Define base directory
merged_dir = "/mnt/weather_data/split_hour_merged/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    print(f"🔹 Processing file: {file_path}")

    # Read file
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Rename columns: Remove location name and apply clean headers
    new_columns = ["Date"] + [column_rename_map[col_name.split("_")[-1]] for col_name in df.columns[1:]]

    # Apply renaming
    df_cleaned = df.toDF(*new_columns)

    # Save back to CSV
    df_cleaned.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

    print(f"✅ Renamed and saved: {file_path}")

print("🎯 All merged tables have clean headers!")


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_hour_merged/ALCACOVAS.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
# Define base directory
merged_dir = "/mnt/weather_data/split_hour_merged/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    print(f"🔹 Processing file: {file_path}")

    # Read file with headers
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Get the column names before removing rows
    columns = df.columns

    # Remove first two rows (skip metadata but keep headers)
    df_cleaned = df.rdd.zipWithIndex().filter(lambda row: row[1] > 1).map(lambda row: row[0]).toDF(df.schema)

    # Apply the correct column names again
    df_cleaned = df_cleaned.toDF(*columns)

    # Save back to CSV with headers
    df_cleaned.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

    print(f"✅ Cleaned and saved: {file_path}")

print("🎯 All merged tables now have the first two rows removed, but headers remain intact!")


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_hour_merged/BATALHA.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
location_coordinates = {
    "ALAGOA/": (39.6833, -7.7667),
    "ALCACOVAS/": (38.3833, -8.0167),
    "BARRAGEM_DE_CASTELO_BURGES/": (39.0833, -9.0833),
    "REBORDELO/": (41.733, -7.167),
    "BARRAGEM_DO_DIVOR/": (38.7167, -7.8833),
    "BARRAGEM_DO_ROXO/": (37.8833, -8.05),
    "BATALHA/": (39.6608, -8.8258),
    "CAMPO_EXPERIMENTAL_CRATO/": (39.2833, -7.65),
    "CAXARIAS/": (39.6833, -8.5),
    "COLARES/": (38.8, -9.45),
    "COMPORTA/": (38.3833, -8.7833),
    "VILA_NOVA_DE_CERVEIRA/": (41.9401, -8.74444),
    "GONDIZALVES/": (41.5667, -8.4167),
    "JUNQUEIRA/": (41.5333, -8.6),
    "MINAS_DE_JALES/": (41.5167, -7.5),
    "PROENCA_A_NOVA/": (39.75, -7.9167),
    "SANTAREM/": (39.2333, -8.6833),
    "SAO_BRAS_DE_ALPORTEL/": (37.15, -7.8833)
}


Add `Latitude` and `Longitude` columns to each merged file, matching on station name.

In [0]:
from pyspark.sql.functions import lit

# Define base directory
merged_dir = "/mnt/weather_data/split_hour_merged/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    location_name = file.name.replace(".csv", "")  # Extract location name from filename

    # Check if we have coordinates for this location
    if location_name in location_coordinates:
        latitude, longitude = location_coordinates[location_name]

        print(f"🔹 Adding coordinates to {location_name}...")

        # Read dataset
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

        # Add Latitude and Longitude columns
        df_with_coords = df.withColumn("Latitude", lit(latitude)).withColumn("Longitude", lit(longitude))

        # Save updated dataset
        df_with_coords.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

        print(f"✅ Updated and saved: {file_path}")
    else:
        print(f"⚠️ No coordinates found for {location_name}, skipping.")

print("🎯 All datasets now include coordinates!")


In [0]:
df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/split_hour_merged/SANTAREM.csv")
df_check.display()


## 4. Convert to Delta and register tables

Clean the column names, write each station as a Delta table, and register the tables in
the metastore.

In [0]:
# Function to clean column names for Delta tables
def clean_column_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("%", "percent")
        .replace(",", "")
        .replace(";", "")   # ← add this line
    )

# Define base directories
merged_dir = "/mnt/weather_data/split_hour_merged/"
delta_base_path = "/mnt/weather_data/hourly/delta/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    location_name = file.name.replace(".csv", "")  # Extract location name

    print(f"🔹 Converting {location_name} to Delta format...")

    # Read CSV
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Clean column names
    new_columns = [clean_column_name(col_name) for col_name in df.columns]
    df_cleaned = df.toDF(*new_columns)

    # Define Delta table path
    delta_path = f"{delta_base_path}{location_name}"

    # Write to Delta format
    df_cleaned.write.format("delta").option("mergeSchema", "true").mode("overwrite").save(delta_path)

    print(f"✅ Delta table saved at: {delta_path}")

print("🎯 All datasets are now saved in Delta format with cleaned column names!")


In [0]:
display(dbutils.fs.ls("/mnt/weather_data/hourly/delta/"))

In [0]:
df_check = spark.read.format("delta").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/hourly/delta/SANTAREM/")
df_check.display()


In [0]:
df_check = spark.read.format("delta").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/hourly/delta/ALAGOA/")
df_check.display()


Register each Delta table in the metastore (initially in the `default` schema).

In [0]:
# Function to clean table names for Hive Metastore
def clean_table_name(name):
    name = name.rstrip("/")  # Remove trailing slashes
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)  # Replace invalid characters with "_"
    return name.lower()  # Convert to lowercase for consistency

# Define Hive table name (adjust schema/database if needed)
for file in dbutils.fs.ls(delta_base_path):
    location_name = clean_table_name(file.name)  # Clean table name

    print(f"🔹 Saving as Hive table: {location_name}...")

    # Read Delta table
    df = spark.read.format("delta").load(f"{delta_base_path}{file.name}")

    # Save as Hive table
    df.write.format("delta").mode("overwrite").saveAsTable(location_name)

    print(f"✅ Hive table created: {location_name}")

print("🎯 All Delta tables are now registered in the Hive Metastore!")


Move the tables from `default` into the `weather_data` schema.

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS weather_data;

In [0]:
%sql
CREATE OR REPLACE TABLE weather_data.alagoa AS SELECT * FROM default.alagoa;
DROP TABLE default.alagoa;

CREATE OR REPLACE TABLE weather_data.alcacovas AS SELECT * FROM default.alcacovas;
DROP TABLE default.alcacovas;

CREATE OR REPLACE TABLE weather_data.barragem_de_castelo_burgoes AS SELECT * FROM default.barragem_de_castelo_burges;
DROP TABLE default.barragem_de_castelo_burges;

CREATE OR REPLACE TABLE weather_data.rebordelo AS SELECT * FROM default.rebordelo;
DROP TABLE default.rebordelo;

CREATE OR REPLACE TABLE weather_data.barragem_do_divor AS SELECT * FROM default.barragem_do_divor;
DROP TABLE default.barragem_do_divor;

CREATE OR REPLACE TABLE weather_data.barragem_do_roxo AS SELECT * FROM default.barragem_do_roxo;
DROP TABLE default.barragem_do_roxo;

CREATE OR REPLACE TABLE weather_data.batalha AS SELECT * FROM default.batalha;
DROP TABLE default.batalha;

CREATE OR REPLACE TABLE weather_data.campo_experimental_crato AS SELECT * FROM default.campo_experimental_crato;
DROP TABLE default.campo_experimental_crato;

CREATE OR REPLACE TABLE weather_data.caxarias AS SELECT * FROM default.caxarias;
DROP TABLE default.caxarias;

CREATE OR REPLACE TABLE weather_data.colares AS SELECT * FROM default.colares;
DROP TABLE default.colares;

CREATE OR REPLACE TABLE weather_data.comporta AS SELECT * FROM default.comporta;
DROP TABLE default.comporta;

CREATE OR REPLACE TABLE weather_data.vila_nova_de_cerveira AS SELECT * FROM default.vila_nova_de_cerveira;
DROP TABLE default.vila_nova_de_cerveira;

CREATE OR REPLACE TABLE weather_data.gondizalves AS SELECT * FROM default.gondizalves;
DROP TABLE default.gondizalves;

CREATE OR REPLACE TABLE weather_data.junqueira AS SELECT * FROM default.junqueira;
DROP TABLE default.junqueira;

CREATE OR REPLACE TABLE weather_data.minas_de_jales AS SELECT * FROM default.minas_de_jales;
DROP TABLE default.minas_de_jales;

CREATE OR REPLACE TABLE weather_data.proenca_a_nova AS SELECT * FROM default.proenca_a_nova;
DROP TABLE default.proenca_a_nova;

CREATE OR REPLACE TABLE weather_data.santarem AS SELECT * FROM default.santarem;
DROP TABLE default.santarem;

CREATE OR REPLACE TABLE weather_data.sao_bras_de_alportel AS SELECT * FROM default.sao_bras_de_alportel;
DROP TABLE default.sao_bras_de_alportel;

